In [4]:
!pip install -r requirements.txt
from warpdrive import WarpDrive
import requests
import random
from faker import Faker
from datetime import datetime, timedelta
import json

fake = Faker()

# Sample prompt templates and topics
templates = [
    "Explain {topic} like I'm five.",
    "Write a short story about {topic}.",
    "Give an example of {topic} in real life.",
    "What's the opposite of {topic}?",
    "Summarize a debate about {topic}."
]

topics = ["blockchain", "gravity", "photosynthesis", "climate change", "AI", "inflation"]

# Fake responses (in practice, you might use a real LLM for these)
responses = [
    "It's a process plants use to make food.",
    "A chain of computers that work together.",
    "It makes things fall to the ground.",
    "It affects how much things cost over time.",
    "A hot topic with many opinions!"
]

# Generate a list of Purview-style logs
def generate_logs(n=10):
    logs = []
    now = datetime.utcnow()
    for _ in range(n):
        prompt = random.choice(templates).format(topic=random.choice(topics))
        response = random.choice(responses)
        timestamp = (now - timedelta(seconds=random.randint(0, 10000))).isoformat() + "Z"
        log_entry = {
            "prompt": prompt,
            "response": response,
            "timestamp": timestamp
        }
        logs.append(log_entry)
    return logs

# Create and export logs
json_logs = generate_logs(5)
print(json.dumps(json_logs, indent=2))



wd = WarpDrive()


username = wd.get_args("username")
password = wd.get_args("password")
pipeline_id = wd.get_args("pipeline_id")
project_id = wd.get_args("project_id")
model_id = wd.get_args("model_id")
model_family = wd.get_args("model_family")

url = "https://nimbus-uno-qa.nginx.solyticspartners.com/genai/llm/biasness-metrics"

Org = "qa"
# your code here
login_headers = {
            'Content-Type': 'application/x-www-form-urlencoded',
            'Org': Org
        }
login_payload = {
            'grant_type': 'password',
            'scope': 'openid',
            'client_id': 'Nimbus',
            'username': username,
            'password': password
        }
login_url = "https://nimbus-uno-qa-keycloak.solyticspartners.com/realms/qa/protocol/openid-connect/token"
login_response = requests.post(url= login_url, headers = login_headers, data = login_payload)
token = 'Bearer ' + login_response.json()["access_token"]
print(token)


def convert_json_to_prompt_data(json_data, latency=None):
    """
    Convert a list of dicts with 'prompt', 'response', and 'timestamp' keys
    to prompt_data format with optional latency added.
    """
    return [
        {
            "prompt": entry.get("prompt", ""),
            "response": entry.get("response", ""),
            "timestamp": entry.get("timestamp", ""),
            "latency": latency
        }
        for entry in json_data
    ]

prompt_data = convert_json_to_prompt_data(json_logs, latency=0.25)


headers = {
        "Authorization": token,
        "Org": Org,
        "Content-Type": "application/json"
    }

payload = {
        "project_id": project_id,
        "pipeline_id": pipeline_id,
        "model_family": model_family,
        "selected_models": [model_id],
        "metrics": ["toxicity_detox", "polarity"],
        "prompt_data": prompt_data
        
    }




response = requests.post(url = url, headers=headers, json=payload)

if response.status_code == 200:
    print(response.json())
else:
    response.raise_for_status()



    
    
    






[
  {
    "prompt": "Summarize a debate about photosynthesis.",
    "response": "A chain of computers that work together.",
    "timestamp": "2025-05-30T10:52:06.041718Z"
  },
  {
    "prompt": "Explain gravity like I'm five.",
    "response": "A chain of computers that work together.",
    "timestamp": "2025-05-30T10:26:01.041718Z"
  },
  {
    "prompt": "Summarize a debate about blockchain.",
    "response": "It affects how much things cost over time.",
    "timestamp": "2025-05-30T10:58:41.041718Z"
  },
  {
    "prompt": "Give an example of climate change in real life.",
    "response": "A chain of computers that work together.",
    "timestamp": "2025-05-30T09:49:50.041718Z"
  },
  {
    "prompt": "Give an example of climate change in real life.",
    "response": "It affects how much things cost over time.",
    "timestamp": "2025-05-30T09:21:41.041718Z"
  }
]
Bearer eyJhbGciOiJSUzI1NiIsInR5cCIgOiAiSldUIiwia2lkIiA6ICIyeTF5VGlTSlJDdzBWdDVuT0dkX1lCLWRLN2FDel9CT0d5NXVIbjlRSWtNIn0.eyJl